# 04 · Producción agrícola: silver y gold
**Responsable:** P2 · **Código:** `src/silver_agricola.py`, `src/gold_agricola.py` · **Salidas:** `silver/agricola.parquet`, `gold/cultivo_municipio.parquet`, figura por cultivo

**Qué hace (silver)**
- Convierte números (acepta `1234.5` y `1.234,5`) y cuenta los valores que no se pudieron leer, antes de convertirlos.
- Reglas que **rechazan:** departamento distinto de 15, código inválido, periodo mal escrito, producción o área faltantes (faltante ≠ cero), valores negativos y filas repetidas por municipio-cultivo-periodo-estado.
- Reglas que **solo alertan:** área cosechada > sembrada, producción sin área cosechada y rendimientos extremos. Se marcan en columnas `alerta_*` y no se borran.

**Qué hace (gold)**
- Suma los periodos A y B por municipio y cultivo. El rendimiento es **Σ producción / Σ área cosechada**, nunca el promedio de rendimientos. Si el área es 0, el rendimiento queda null.
- Concilia: la producción total de gold debe ser igual a la de silver.

**Advertencia:** sumar toneladas de cultivos distintos (papa fresca, café pergamino, caña) no da un valor económico.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import polars as pl
from src.config import *
asegurar_carpetas()

In [2]:
from src.silver_agricola import construir_agricola
agr = construir_agricola()
agr.group_by("periodo").len()

[silver/agricola] revisados=2613 validos=2613 rechazados=0


periodo,len
str,u32
"""2023A""",896
"""2023B""",842
"""2023""",875


In [3]:
from src.gold_agricola import construir_cultivo_municipio, resumen_departamental_por_cultivo
cm = construir_cultivo_municipio(agr)
resumen = resumen_departamental_por_cultivo(cm)
resumen.head(15)

[gold/cultivo_municipio] revisados=1731 validos=1731 rechazados=0


cultivo,estado_fisico,produccion_t,area_cosechada_ha,municipios,rendimiento_t_ha
str,str,f64,f64,u32,f64
"""Caña""","""Caña o verde""",2.0783e6,22344.949,60,93.011981
"""Papa""","""En fresco""",962643.34,46526.48,92,20.690225
"""Tomate""","""En fresco""",183195.63,2282.28,51,80.268692
"""Cebolla de bulbo""","""En fresco""",132726.58,4845.8,48,27.390024
"""Cebolla de rama""","""En fresco""",91066.52,2596.9,7,35.067396
…,…,…,…,…,…
"""Tomate de árbol""","""En fresco""",16200.025,982.85,50,16.482703
"""Mora""","""En fresco""",15083.255,1226.77,52,12.295096
"""Ciruela""","""En fresco""",13263.835,888.83,17,14.922803


In [4]:
from src import graficos
graficos.produccion_rendimiento_por_cultivo(resumen.to_pandas(), ANIO_EVA)

PosixPath('/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/proyecto_boyaca/outputs/figures/p2_produccion_rendimiento_cultivos_2023.png')

In [5]:
pl.read_csv(QUALITY / "metricas_silver.csv").filter(pl.col("fuente") == "agricola")

fecha_ejecucion,capa,fuente,regla,descripcion,revisados,afectados,resultado,accion
str,str,str,str,str,i64,i64,str,str
"""2026-11-05T12:51:12""","""silver""","""agricola""","""area_sembrada_ha_no_numerico""","""Valores de area_sembrada_ha qu…",2613,0,"""OK""","""alertar"""
"""2026-11-05T12:51:12""","""silver""","""agricola""","""area_cosechada_ha_no_numerico""","""Valores de area_cosechada_ha q…",2613,0,"""OK""","""alertar"""
"""2026-11-05T12:51:12""","""silver""","""agricola""","""produccion_t_no_numerico""","""Valores de produccion_t que no…",2613,0,"""OK""","""alertar"""
"""2026-11-05T12:51:12""","""silver""","""agricola""","""rendimiento_fuente_t_ha_no_num…","""Valores de rendimiento_fuente_…",2613,0,"""OK""","""alertar"""
"""2026-11-05T12:51:12""","""silver""","""agricola""","""departamento_boyaca""","""Solo filas de Boyacá (15)""",2613,0,"""OK""","""rechazar"""
…,…,…,…,…,…,…,…,…
"""2026-11-05T12:51:12""","""silver""","""agricola""","""produccion_con_area""","""Si hay producción, hay área co…",2613,0,"""OK""","""alertar"""
"""2026-11-05T12:51:12""","""silver""","""agricola""","""rendimiento_fuente_razonable""","""Rendimiento reportado entre 0 …",2613,0,"""OK""","""alertar"""
"""2026-11-05T12:51:12""","""silver""","""agricola""","""clave_unica""","""Una fila por municipio, cultiv…",2613,0,"""OK""","""rechazar"""
